# DATA103 Phase 1: Problem Formulation and Exploratory Data Analysis
## Automated American Sign Language (ASL) Fingerspelling Recognition via Hand Gesture Classification

**Course:** DATA103 - Introduction to Machine Learning \
**Term / Academic Year:** Term 1, A.Y. 2026-2027 \
**Instructor:** Dr. John Jethro Virtusio \
**Authors:** Conda, Gopico, Marqueses, Piczon \
**Dataset Source:** [Sign Language MNIST on Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)

The GitHub repository for this project is available [here](https://github.com/lowestofthe1ow/data103-final-project/).

In [ ]:
# Environment setup and library imports
import string
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Visual formatting configurations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Core libraries imported successfully.")

## 1. Background of the study

### 1.1 Motivation and significance

[The World Federation of the Deaf estimates that over 70 million individuals globally rely on sign language as their primary mode of communication](https://wfdeaf.org/contact/faqs/). This creates accessibility barriers in various environments where non-deaf individuals are unlikely to understand it, including (but not limited to) healthcare and education. While truly fluent sign language involves continuous two-handed gestures and facial expressions, *fingerspelling* (i.e., spelling out words letter-by-letter) is crucial for spelling non-standard words like personal names and technical jargon. Developing an automated, lightweight fingerspelling classifier can help with real-time transcription on resource-constrained consumer devices without requiring specialized hardware.

### 1.2 Research objectives

This study investigates the performance of classical machine learning algorithms from the DATA103 curriculum on image-based gesture recognition:

1. To what extent can linear decision boundaries (logistic regression) adequately separate high-dimensional ($d = 784$ features) flattened pixel representations? Are non-linear models (decision trees, random forests, multi-layer perceptrons) fundamentally required to resolve visual ambiguities?
2. Does dimensional reduction on the feature set (e.g. PCA) shrink the gap between linear and non-linear methods, and to what extent?
3. Which specific gesture pairs (e.g., gestures with closed fists such as `'A'`, `'E'`, `'M'`, `'N'`, or `'S'`) exhibit the highest confusion? Can distance-based neighborhood models (KNN) mitigate these classification errors?
4. How does predictive performance trade off against model size (e.g., parameter count) across the model families?

### 1.3 Hypotheses

$H1_0 :$ Non-linear methods achieve a test macro-F1 no higher than logistic regression. \
$H1_1 :$ Non-linear methods achieve higher test macro-F1 than logistic regression.

$H2_0 :$ Feature dimensionality reduction does not significantly shrinks the macro-F1 gap between logistic regression and non-linear methods. \
$H2_1 :$ Feature dimensionality reduction significantly shrinks the macro-F1 gap between logistic regression and non-linear methods.

$H3_0 :$ Misclassifications are spread broadly across letter pairs, or KNN's mean per-class F1 on the most-confused letters is no higher than that of logistic regression. \
$H3_1 :$ For each model, misclassifications concentrate around the most-confused letter pairs, and KNN achieves a higher mean per-class F1 than logistic regression.

$H4_0 :$ Higher-capacity models do not achieve significantly higher test macro-F1 than smaller models (accounting for the difference model size). \
$H4_1 :$ Higher-capacity models achieve significantly higher test macro-F1 than smaller models (accounting for the difference model size).

## 2. Dataset and exploratory data analysis

We use the [Sign Language MNIST](https://www.kaggle.com/datasets/datamunge/sign-language-mnist) dataset, which follows the standard MNIST benchmark structure. Each row represents an individual, pre-centered, $28 \times 28$ grayscale image of an American Sign Language (ASL) hand gesture that corresponds to a single letter.

* **Column 0 (`label`):** The numerical target identifier representing the letter.
* **Columns 1 to 784 (`pixel1` to `pixel784`):** Raw grayscale brightness values ranging from `0` (pure black) to `255` (pure white).

In [ ]:
# Load from local directory
train_path = "data/sign_mnist_train.csv"
test_path = "data/sign_mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Training dataset shape: {train_df.shape} (rows, columns)")
print(f"Testing dataset shape:  {test_df.shape} (rows, columns)")

In [ ]:
# Check for nulls
train_nulls = train_df.isnull().sum().sum()
test_nulls = test_df.isnull().sum().sum()

print(f"Total null/missing values in train set: {train_nulls}")
print(f"Total null/missing values in test set:  {test_nulls}")

# Check pixel intensity ranges
pixel_min = train_df.iloc[:, 1:].min().min()
pixel_max = train_df.iloc[:, 1:].max().max()

print(f"Pixel intensity range: [{pixel_min}, {pixel_max}]")

# Preview initial records and columns (first 8 columns only to fit screen)
train_df.iloc[:5, :8]

### 2.1. Class labels

The dataset contains **24 distinct classes** from the 26 letters of the English alphabet. The letters `'J'` (index `9`) and `'Z'` (index `25`) are omitted from the dataset because their production in ASL requires continuous hand trajectory/motion paths, making them unsuitable for static 2D image classification.

Labels represent class indices: $A = 0, B = 1, \dots, Y = 24$. We observe that the training data demonstrates a well-balanced distribution across all 24 classes.

In [ ]:
# Skip J and Z
alphabet = list(string.ascii_uppercase)
valid_letters = [char for char in alphabet if char not in ["J", "Z"]]
unique_labels = sorted(train_df["label"].unique())

label_to_letter = {lbl: valid_letters[i] for i, lbl in enumerate(unique_labels)}

# Show class distribution
plt.figure(figsize=(12, 4.5))
ax = sns.countplot(x="label", data=train_df, color="teal", edgecolor="black", linewidth=0.5)
plt.title("Training Set Class Distribution", fontsize=13, weight="bold")
plt.xlabel("Class", fontsize=11)
plt.ylabel("Count", fontsize=11)
plt.xticks(ticks=range(len(unique_labels)), labels=[label_to_letter[lbl] for lbl in unique_labels])
plt.tight_layout()
plt.show()

min_count = train_df["label"].value_counts().min()
max_count = train_df["label"].value_counts().max()
print(f"Class frequency range:\nMin = {min_count} samples\nMax = {max_count} samples")

### 2.2 Visual EDA

Because each row represents a flattened 784-dimensional vector (28x28 image), we reshape sample rows back into 2D matrices to inspect visual characteristics. Here we show a single sample image from each class:

In [ ]:
# Reshape to matrix and plot one sample sign for each of the 24 classes
X_train_raw = train_df.drop("label", axis=1).values
y_train_raw = train_df["label"].values

fig, axes = plt.subplots(4, 6, figsize=(12, 8))
axes = axes.flatten()

for i, lbl in enumerate(unique_labels):
    sample_idx = np.where(y_train_raw == lbl)[0][0]
    img_matrix = X_train_raw[sample_idx].reshape(28, 28)
    
    axes[i].imshow(img_matrix, cmap="gray")
    axes[i].set_title(f"Class {lbl}: '{label_to_letter[lbl]}'", fontsize=10, weight="bold")
    axes[i].axis("off")

plt.suptitle("Per-Class Sample Images", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

We see that, at a resolution of $28 \times 28$, the closed-fist gestures (`'A'`, `'E'`, `'M'`, `'N'`, and `'S'`) appear to be very similar to each other. Others, like `'R'` and `'U'`, also appear visually similar despite having a different number of fingers raised. To get a better view of the characteristics of entire classes, we show here the mean image intensity heatmap per class:

In [ ]:
# Mean image per class on a shared scale
class_means = {lbl: X_train_raw[y_train_raw == lbl].mean(axis=0).reshape(28, 28)
               for lbl in unique_labels}
vmax = max(m.max() for m in class_means.values())

fig, axes = plt.subplots(4, 6, figsize=(13, 9))
axes = axes.flatten()

for i, lbl in enumerate(unique_labels):
    im = axes[i].imshow(class_means[lbl], cmap="inferno", vmin=0, vmax=vmax)
    axes[i].set_title(f"Class {lbl}: '{label_to_letter[lbl]}'", fontsize=10, weight="bold")
    axes[i].axis("off")

fig.colorbar(im, ax=axes.tolist(), label="Mean Intensity", shrink=0.6)
plt.suptitle("Per-Class Mean Spatial Heatmap", fontsize=14, weight="bold")
plt.show()

Notably, the `'P'` and `'Q'` classes appear to be almost unintelligible compared to the other classes, possibly due to strong variations in photo environment or hand position. Another concern is due to shifting image positions, finer details such as the number of fingers raised may blur into each other and make them more difficult to distinguish. This is again seen in `'R'` and `'U'`, or `'G'` and `'H'`, but is again most prominent (at least, to the human eye) with the closed-fist gestures, (`'A'`, `'E'`, `'M'`, `'N'`, and `'S'`). For reference, we also show here the mean intensity heatmap over the entire dataset (which predictably takes the shape of a fist):

In [ ]:
mean_hand_matrix = X_train_raw.mean(axis=0).reshape(28, 28)

plt.figure(figsize=(5.5, 4.5))
plt.imshow(mean_hand_matrix, cmap="inferno")
plt.colorbar(label="Mean Intensity (0-255)")
plt.title("Aggregate Mean Spatial Heatmap", fontsize=12, weight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

We can take this a step further and show which of the mean image classes are most similar to each other in terms of the Euclidean distance. `'P'` and `'Q'` remain the most problematic, but we note that contrary to expectations here, `'A'`, `'E'`, `'M'`, `'N'`, and `'S'` appear to be fairly distant from each other.

In [ ]:
# Centroid distance heatmap
letters = [label_to_letter[l] for l in unique_labels]
M = np.stack([class_means[l].ravel() for l in unique_labels])
dist = np.linalg.norm(M[:, None] - M[None], axis=-1)

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(dist, mask=np.triu(np.ones_like(dist, dtype=bool)), cmap="mako_r", square=True,
            xticklabels=letters, yticklabels=letters, linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Euclidean distance between class means", "shrink": 0.8}, ax=ax)
ax.set_title("Class Centroid Distances", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

# Ten closest pairs
i, j = np.tril_indices(len(letters), -1)
for k in np.argsort(dist[i, j])[:10]:
    print(f"{letters[i[k]]}-{letters[j[k]]}: {dist[i[k], j[k]]:.1f}")

We can see this more numerically by plotting a ridgeline plot for per-class pixel intensity distributions. Again, the classes `'P'` and `'Q'` stand out as pixel intensities appear to be much more spread out for them compared to the other classes. This may imply that classifiers could struggle differentiating them the most.

In [ ]:
# Ridgeline of pixel intensity per class
bins = np.linspace(0, 256, 33)
centers = (bins[:-1] + bins[1:]) / 2

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette("mako", len(unique_labels))

for i, lbl in enumerate(unique_labels):
    h, _ = np.histogram(X_train_raw[y_train_raw == lbl].ravel(), bins=bins, density=True)
    h = h / h.max() * 1.8
    y0 = len(unique_labels) - i
    ax.fill_between(centers, y0, y0 + h, color=colors[i], alpha=0.85, edgecolor="white", linewidth=0.8, zorder=i)

ax.set_yticks(np.arange(len(unique_labels), 0, -1))
ax.set_yticklabels([label_to_letter[l] for l in unique_labels])
ax.set_title("Pixel Intensity Distribution per Gesture Class", fontsize=13, weight="bold")
ax.set_xlabel("Pixel Intensity (0-255)", fontsize=11)
ax.set_ylabel("Gesture Letter Class", fontsize=11)
sns.despine(left=True)
plt.tight_layout()
plt.show()

## 5. Planned Phase 2 methodology

Following Phase 1 problem formulation and data validation, we will implement Phase 2 as follows:

### 5.1 Preprocessing

1. *Min-max normalization*. We will scale all 784 pixel intensity inputs from the $[0, 255]$ integer range to floating-point values in $[0.0, 1.0]$. This is critical to ensure distance metrics in KNN and gradient updates in, e.g., MLP, remain stable and well-conditioned.
2. *Principal Component Analysis (PCA)*. We apply feature dimensionality reduction for our second research objective. PCA can help speed up approaches like KNN, and we could use it for easier 2D or 3D visualization as well. **Note that our final choice of method here may not be restricted to PCA only**.

### 5.2 Algorithms

We will implement and benchmark candidate models covered in the course curriculum:

- *K-Nearest Neighbors (KNN Classification)*. Non-parametric baseline using Euclidean distance across flattened 784-dimensional space.
- *Regularized Multinomial Logistic Regression*. Linear baseline utilizing L1 and L2 penalty terms to prevent overfitting across the 784 collinear pixel features.
- *Decision Trees & Random Forest Classifier*. Non-linear axis-aligned partitioning and bagging ensembles to assess feature importance across pixel coordinates.
- *Multi-Layer Perceptron (MLP)*. Fully connected feedforward neural network baseline utilizing backpropagation.

### 5.3 Model evaluation

- *Train-test-split*. We will retain the official `sign_mnist_test.csv` (7,172 samples) strictly for final evaluation. However, we will keep a 15% stratified validation subset from `sign_mnist_train.csv` for hyperparameter tuning and cross-validation (with a standard $k = 10$ folds).
- *Macro F1-score & accuracy*. Balanced evaluation across all 24 classes.
- *Confusion matrix*. Systematic error analysis to measure inter-class confusion between morphologically similar hand signs (e.g., `'E'` vs. `'S'`).
